In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST Letters: 145,600 chars, 26 balanced classes (A-Z, case-merged)
#  ~5,600 samples per class — largest balanced per-class count in EMNIST
#  NOTE: tfds labels are 1-indexed (1=A … 26=Z); we remap to 0-indexed below
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "emnist/letters"
NUM_CLASSES  = 26
IMG          = 28
BS           = 64
EPOCHS       = 50            # fewer epochs: easy task + balanced data
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.0           # balanced dataset — no smoothing needed
DROP_PATH    = 0.05
WARMUP_EP    = 4
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/letters"
AUTOTUNE     = tf.data.AUTOTUNE

LABELS = [chr(c) for c in range(65, 91)]   # ['A', 'B', ..., 'Z']

os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f"[INFO] Loading {DATASET} via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING  —  remap 1-indexed labels → 0-indexed
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32) - 1          # 1-26  →  0-25
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.25)
    image = tf.image.random_contrast(image, 0.75, 1.25)
    image = tf.pad(image, [[3, 3], [3, 3], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    # NOTE: no horizontal flip — 'b'/'d', 'p'/'q' confusions are real
    image = image + tf.random.normal(tf.shape(image), stddev=0.03)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(10_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds    = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BS).prefetch(AUTOTUNE)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x): return tf.nn.gelu(x)

class DropPath(keras.layers.Layer):
    def __init__(self, drop_prob=0.0, **kwargs):
        super().__init__(**kwargs); self.drop_prob = drop_prob
    def call(self, x, training=None):
        if not training or self.drop_prob == 0.0: return x
        keep  = 1.0 - self.drop_prob
        shape = (tf.shape(x)[0],) + (1,) * (len(x.shape) - 1)
        mask  = tf.floor(keep + tf.random.uniform(shape, dtype=x.dtype))
        return x * mask / keep
    def get_config(self):
        cfg = super().get_config(); cfg["drop_prob"] = self.drop_prob; return cfg

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels, drop_rate=0.0):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if drop_rate > 0: x = DropPath(drop_rate)(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch, drop_rate=0.0):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch, drop_rate)
    r2  = residual_block(r1, out_ch, drop_rate)
    r3  = residual_block(r2, out_ch, drop_rate)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def english_topology_module(x, out_features):
    """Letters-only: curves (c,e,o,s), verticals (i,l,t), ascenders (b,d,f,h,k)."""
    h  = layers.Conv2D(48, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(48, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    d4 = layers.Conv2D(32, 3, padding="same", dilation_rate=4, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2, d4])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    return layers.Dense(out_features, activation=gelu)(out)

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL  —  compact head for 26-class task
# ─────────────────────────────────────────────────────────────────────────────
def build_model(num_classes=26, image_size=28, drop_path_rate=0.05):
    K, DR = num_classes, drop_path_rate
    inp = Input(shape=(image_size, image_size, 1), name="input")

    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    enc1 = dense_res_block(stem, 64,  64,  drop_rate=DR * 0.5)
    enc2 = dense_res_block(enc1, 64,  128, drop_rate=DR)
    enc3 = dense_res_block(enc2, 128, 256, drop_rate=DR)

    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    multi_scale = layers.Concatenate()([gap1, gap2, gap3])

    stm_out = english_topology_module(enc3, 256)

    fused = layers.Concatenate()([stm_out, multi_scale])
    x   = layers.Dense(128, use_bias=False)(fused)     # narrower head for 26 classes
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    x   = layers.Dropout(0.35)(x)
    out = layers.Dense(K, name="logits")(x)
    return Model(inputs=inp, outputs=out, name="WhatNet_Letters")

# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model        = build_model(NUM_CLASSES, IMG, DROP_PATH)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15,
                      restore_best_weights=True, verbose=1),
    ],
)

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_LETTERS", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

2026-04-22 06:43:16.372766: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776840196.590800      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776840196.660590      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776840197.188943      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776840197.188985      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776840197.188988      55 computation_placer.cc:177] computation placer alr

[INFO] Loading emnist/letters via tensorflow_datasets ...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/letters/incomplete.HBRPOI_3.1.0/emnist-test.tfrecord*...:   0%|    …

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/letters/3.1.0. Subsequent calls will reuse this data.
[INFO] Train: 79,920 | Val: 8,880 | Test: 14,800
[INFO] Steps/epoch: 1248
[INFO] Params: 5,577,962
Epoch 1/50


I0000 00:00:1776840323.789867     133 service.cc:152] XLA service 0x312d9e10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776840323.789921     133 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776840323.789927     133 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776840328.212138     133 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776840353.454475     133 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1249/1249 ━━━━━━━━━━━━━━━━━━━━ 188s 108ms/step - accuracy: 0.1211 - loss: 3.2246 - val_accuracy: 0.8427 - val_loss: 0.5958
Epoch 2/50
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.8429 - loss: 0.5713 - val_accuracy: 0.9020 - val_loss: 0.2848
Epoch 3/50
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9086 - loss: 0.2982 - val_accuracy: 0.9091 - val_loss: 0.2543
Epoch 4/50
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9216 - loss: 0.2450 - val_accuracy: 0.8972 - val_loss: 0.2916
Epoch 5/50
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 95s 76ms/step - accuracy: 0.9277 - loss: 0.2220 - val_accuracy: 0.9348 - val_loss: 0.1812
Epoch 6/50
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 95s 76ms/step - accuracy: 0.9370 - loss: 0.1889 - val_accuracy: 0.9338 - val_loss: 0.1860
Epoch 7/50
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 95s 76ms/step - accuracy: 0.9405 - loss: 0.1760 - val_accuracy: 0.9381 - val_loss: 0.1712
Epoch 8/50
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9441 - loss: 0.

# Method

In [1]:
import os, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST Letters: 145,600 chars, 26 balanced classes (A-Z)
#  ~5,600 samples/class. Case-merged: 'A' and 'a' share one label.
#  NOTE: tfds labels are 1-indexed (1=A … 26=Z); remapped to 0-indexed below.
#  NO horizontal flip — b/d and p/q confusions are real in case-merged data.
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "emnist/letters"
NUM_CLASSES  = 26
IMG          = 28
BS           = 64
EPOCHS       = 100
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1          # case-merging introduces genuine label ambiguity
WARMUP_EP    = 4
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/letters"
AUTOTUNE     = tf.data.AUTOTUNE

LABELS = [chr(c) for c in range(65, 91)]   # ['A', 'B', ..., 'Z']
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f"[INFO] Loading {DATASET} via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING  —  remap 1-indexed labels → 0-indexed
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32) - 1          # 1-26  →  0-25
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.25)
    image = tf.image.random_contrast(image, 0.75, 1.25)
    image = tf.pad(image, [[3, 3], [3, 3], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    # NO horizontal flip — b/d and p/q confusions are real
    image = image + tf.random.normal(tf.shape(image), stddev=0.03)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(10_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BS).prefetch(AUTOTUNE)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x): return tf.nn.gelu(x)

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    # r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2]) #r3
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def adaptive_filter_capsule(x, num_classes, capsule_dim=32):
    """
    Lightweight capsule routing.
    capsule_dim=32 (vs 16 for digits) — 26-class case-merged letters have
    higher intra-class variance so richer capsule space helps.
    """
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps = layers.Multiply()([x_sliced, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    return layers.BatchNormalization()(caps)

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────────────────
def build_model(num_classes=26, image_size=28):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem: 3-path (texture + horizontal + vertical) ────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64,  64)
    enc2 = dense_res_block(enc1, 64,  128)
    enc3 = dense_res_block(enc2, 128, 256)

    # ── Multi-scale GAP fusion ────────────────────────────────────────────
    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])  # (B, 512)

    # ── Adaptive Filter Capsule (capsule_dim=32 for letter complexity) ────
    afc_out = adaptive_filter_capsule(fused_gap, K, capsule_dim=32)   # (B, K)

    # ── Dense head ────────────────────────────────────────────────────────
    x = layers.Dense(128, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(K, name="head_logits")(x)

    # ── Gated fusion: AFC + dense head ────────────────────────────────────
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])
    out = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=out, name="WhatNet_Letters")

# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model        = build_model(NUM_CLASSES, IMG)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15,
                      restore_best_weights=True, verbose=1),
    ],
)

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_LETTERS", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

2026-04-26 05:56:46.726385: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777183006.962937      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777183007.036131      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777183007.569851      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777183007.569899      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777183007.569905      55 computation_placer.cc:177] computation placer alr

[INFO] Loading emnist/letters via tensorflow_datasets ...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/letters/incomplete.HBRPOI_3.1.0/emnist-test.tfrecord*...:   0%|    …

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/letters/3.1.0. Subsequent calls will reuse this data.
[INFO] Train: 79,920 | Val: 8,880 | Test: 14,800
[INFO] Steps/epoch: 1248
[INFO] Params: 5,496,316
Epoch 1/100


I0000 00:00:1777183128.570941     132 service.cc:152] XLA service 0x7f33a8131670 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777183128.571011     132 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777183128.571017     132 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777183132.657203     132 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777183153.136444     132 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1249/1249 ━━━━━━━━━━━━━━━━━━━━ 165s 95ms/step - accuracy: 0.2048 - loss: 2.9525 - val_accuracy: 0.8868 - val_loss: 1.0359
Epoch 2/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 94s 75ms/step - accuracy: 0.8821 - loss: 1.0734 - val_accuracy: 0.9150 - val_loss: 0.9342
Epoch 3/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 95s 76ms/step - accuracy: 0.9165 - loss: 0.9301 - val_accuracy: 0.9173 - val_loss: 0.8817
Epoch 4/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 96s 76ms/step - accuracy: 0.9264 - loss: 0.8756 - val_accuracy: 0.9244 - val_loss: 0.8469
Epoch 5/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 96s 77ms/step - accuracy: 0.9354 - loss: 0.8350 - val_accuracy: 0.9348 - val_loss: 0.8113
Epoch 6/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 96s 77ms/step - accuracy: 0.9407 - loss: 0.8034 - val_accuracy: 0.9357 - val_loss: 0.8004
Epoch 7/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 96s 77ms/step - accuracy: 0.9465 - loss: 0.7816 - val_accuracy: 0.9387 - val_loss: 0.7934
Epoch 8/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 95s 76ms/step - accuracy: 0.9494 - lo

# Corrected

In [1]:
import os, random, json
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST Letters: 145,600 chars, 26 balanced classes (A-Z)
#  ~5,600 samples/class. Case-merged: 'A' and 'a' share one label.
#  NOTE: tfds labels are 1-indexed (1=A … 26=Z); remapped to 0-indexed below.
#  NO horizontal flip — b/d and p/q confusions are real in case-merged data.
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "emnist/letters"
NUM_CLASSES  = 26
IMG          = 28
BS           = 64
EPOCHS       = 100
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1        # reduced from 0.1 — was hurting hard classes
WARMUP_EP    = 4
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/letters"
AUTOTUNE     = tf.data.AUTOTUNE
LABELS = [chr(c) for c in range(65, 91)]   # ['A', 'B', ..., 'Z']
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f"[INFO] Loading {DATASET} via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING  —  remap 1-indexed labels → 0-indexed
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32) - 1          # 1-26  →  0-25
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.25)
    image = tf.image.random_contrast(image, 0.75, 1.25)
    image = tf.pad(image, [[3, 3], [3, 3], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    # NO horizontal flip — b/d and p/q confusions are real
    image = image + tf.random.normal(tf.shape(image), stddev=0.03)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

# FIX: shuffle the full dataset before splitting to avoid biased val set
train_raw = train_raw.shuffle(total, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x): return tf.nn.gelu(x)

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def adaptive_filter_capsule(x, num_classes, capsule_dim=32):
    """
    FIX: replaced broken slice `t[:, :, :capsule_dim]` on the 512-dim fused
    vector with a proper Dense projection to capsule_dim.  The old code only
    used the first 32 of 512 features and caused zero-TP for classes 20-25.
    """
    # Capsule prediction matrix
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)

    # FIX: project x to capsule_dim via Dense, not slicing
    x_proj = layers.Dense(capsule_dim, use_bias=False)(x)          # (B, capsule_dim)
    x_exp  = layers.RepeatVector(num_classes)(x_proj)              # (B, num_classes, capsule_dim)

    caps = layers.Multiply()([x_exp, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)  # (B, num_classes)
    return layers.BatchNormalization()(caps)

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────────────────
def build_model(num_classes=26, image_size=28):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem: 3-path (texture + horizontal + vertical) ────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)
    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64,  64)
    enc2 = dense_res_block(enc1, 64,  128)
    enc3 = dense_res_block(enc2, 128, 256)

    # ── Multi-scale GAP fusion ────────────────────────────────────────────
    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])  # (B, 512)

    # ── Adaptive Filter Capsule (fixed) ───────────────────────────────────
    afc_out = adaptive_filter_capsule(fused_gap, K, capsule_dim=32)   # (B, K)

    # ── Dense head ────────────────────────────────────────────────────────
    x = layers.Dense(128, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dropout(0.30)(x)   # reduced from 0.35
    x = layers.Dense(K, name="head_logits")(x)

    # ── Gated fusion: AFC + dense head ────────────────────────────────────
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])
    out = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=out, name="WhatNet_Letters")

# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model        = build_model(NUM_CLASSES, IMG)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15,
                      restore_best_weights=True, verbose=1),
    ],
)

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
#  FIX: use test_ds (which has preprocess applied → 0-indexed labels) for the
#  manual per-class loop.  The old code used test_ds there too, but the labels
#  coming out of test_ds are already 0-indexed after preprocess, so the loop
#  itself was correct — the real bug was the capsule slicing above.
#  Keeping the loop on test_ds (0-indexed) for clarity and correctness.
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:           # labels are 0-indexed (preprocess applied)
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_LETTERS", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)

print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

2026-04-26 09:48:03.516787: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777196883.748028      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777196883.809070      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777196884.336376      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777196884.336420      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777196884.336424      55 computation_placer.cc:177] computation placer alr

[INFO] Loading emnist/letters via tensorflow_datasets ...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/letters/incomplete.HBRPOI_3.1.0/emnist-test.tfrecord*...:   0%|    …

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/letters/3.1.0. Subsequent calls will reuse this data.
[INFO] Train: 79,920 | Val: 8,880 | Test: 14,800
[INFO] Steps/epoch: 1248
[INFO] Params: 5,512,700
Epoch 1/100


I0000 00:00:1777197008.824159     134 service.cc:152] XLA service 0x466ba0d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777197008.824213     134 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777197008.824217     134 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777197012.717394     134 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777197032.916741     134 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1249/1249 ━━━━━━━━━━━━━━━━━━━━ 172s 98ms/step - accuracy: 0.1716 - loss: 3.0374 - val_accuracy: 0.8842 - val_loss: 1.0340
Epoch 2/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 80ms/step - accuracy: 0.8706 - loss: 1.0907 - val_accuracy: 0.9145 - val_loss: 0.8910
Epoch 3/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9107 - loss: 0.9436 - val_accuracy: 0.9229 - val_loss: 0.8664
Epoch 4/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9235 - loss: 0.8929 - val_accuracy: 0.9231 - val_loss: 0.8580
Epoch 5/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9325 - loss: 0.8552 - val_accuracy: 0.9279 - val_loss: 0.8357
Epoch 6/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 79ms/step - accuracy: 0.9398 - loss: 0.8232 - val_accuracy: 0.9419 - val_loss: 0.7932
Epoch 7/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 79ms/step - accuracy: 0.9441 - loss: 0.7983 - val_accuracy: 0.9444 - val_loss: 0.7855
Epoch 8/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 80ms/step - accuracy: 0.9467 